# 02 Clean and join: build the movies table

- **Reads:** data/raw/ (title.basics, title.ratings, title.crew, name.basics)
- **Writes:** data/processed/movies.parquet
- **Run after:** nothing (needs the Sprint 1 downloads in data/raw/)

What I found: keeping only non-adult movies with 1,000+ votes leaves 49,453 movies, each with its rating, main director and one-hot genre columns.

In [1]:
import csv
import os
import pandas as pd

RAW = "../data/raw"
PROCESSED = "../data/processed"
BASICS = f"{RAW}/title.basics.tsv.gz"
RATINGS = f"{RAW}/title.ratings.tsv.gz"
CREW = f"{RAW}/title.crew.tsv.gz"
NAMES = f"{RAW}/name.basics.tsv.gz"

TSV = dict(sep="\t", na_values="\\N", quoting=csv.QUOTE_NONE)

log = {}

In [2]:
cols = ["tconst", "titleType", "primaryTitle", "isAdult",
        "startYear", "runtimeMinutes", "genres"]

basics = pd.read_csv(
    BASICS,
    **TSV,
    usecols=cols,
    dtype={
        "tconst": "string[pyarrow]",
        "titleType": "category",
        "primaryTitle": "string[pyarrow]",
        "isAdult": "category",
        "startYear": "string[pyarrow]",
        "runtimeMinutes": "string[pyarrow]",
        "genres": "string[pyarrow]",
    },
)
log["1. all titles"] = len(basics)
print(basics.shape)

(12947333, 7)


In [3]:
print(basics["isAdult"].value_counts())
print(basics["isAdult"].cat.categories)

isAdult
0    12528776
1      418557
Name: count, dtype: int64
Index([0, 1], dtype='int64')


In [4]:
is_movie = basics["titleType"] == "movie"
not_adult = basics["isAdult"].astype("string") == "0"

movies = basics[is_movie & not_adult].copy()
del basics

log["2. non-adult movies"] = len(movies)
print(len(movies), "non-adult movies")

751203 non-adult movies


In [5]:
print("Missing before:")
print(movies[["startYear", "runtimeMinutes"]].isna().sum())

movies["startYear"] = pd.to_numeric(movies["startYear"], errors="coerce").astype("Int16")
movies["runtimeMinutes"] = pd.to_numeric(movies["runtimeMinutes"], errors="coerce").astype("Int32")

print("Missing after:")
print(movies[["startYear", "runtimeMinutes"]].isna().sum())

Missing before:
startYear         112536
runtimeMinutes    267272
dtype: int64
Missing after:
startYear         112536
runtimeMinutes    267272
dtype: int64


In [6]:
movies[["startYear", "runtimeMinutes"]].describe()

,startYear,runtimeMinutes
count,638667.0,483931.0
mean,1995.582396,90.017709
std,30.208412,113.944084
min,1894.0,1.0
25%,1978.0,73.0
50%,2009.0,89.0
75%,2018.0,100.0
max,2032.0,51420.0


In [7]:
ratings = pd.read_csv(
    RATINGS,
    **TSV,
    dtype={"tconst": "string[pyarrow]",
           "averageRating": "float32",
           "numVotes": "int32"},
)
print(ratings.shape)
ratings.head()

(1729976, 3)


,tconst,averageRating,numVotes
0,tt0000001,5.7,2239
1,tt0000002,5.4,329
2,tt0000003,6.4,2409
3,tt0000004,5.0,203
4,tt0000005,6.2,3109


In [8]:
print("movies key unique: ", movies["tconst"].is_unique)
print("ratings key unique:", ratings["tconst"].is_unique)

movies key unique:  True
ratings key unique: True


In [9]:
check = movies.merge(
    ratings, on="tconst", how="left",
    validate="one_to_one", indicator=True,
)
check["_merge"].value_counts()

_merge
left_only     404533
both          346670
right_only         0
Name: count, dtype: int64

In [10]:
rated = movies.merge(ratings, on="tconst", how="inner", validate="one_to_one")
log["3. with any rating"] = len(rated)

print(rated["numVotes"].describe())
print("Share with 1,000+ votes:", round((rated["numVotes"] >= 1000).mean(), 3))

count    3.466700e+05
mean     3.770359e+03
std      3.852625e+04
min      5.000000e+00
25%      2.100000e+01
50%      6.600000e+01
75%      3.340000e+02
max      3.262239e+06
Name: numVotes, dtype: float64
Share with 1,000+ votes: 0.143


In [11]:
MIN_VOTES = 1000
rated = rated[rated["numVotes"] >= MIN_VOTES].copy()
log[f"4. {MIN_VOTES:,}+ votes"] = len(rated)
print(len(rated), "movies")

49453 movies


In [12]:
crew = pd.read_csv(
    CREW,
    **TSV,
    usecols=["tconst", "directors"],
    dtype={"tconst": "string[pyarrow]", "directors": "string[pyarrow]"},
)
crew = crew[crew["tconst"].isin(rated["tconst"])]
print(crew.shape)
crew.head()

(49453, 2)


,tconst,directors
570,tt0000574,nm0846879
2105,tt0002130,"nm0078205,nm0655824,nm0209738"
2396,tt0002423,nm0523932
2817,tt0002844,nm0275421
2986,tt0003014,nm0803705


In [13]:
n_dirs = crew["directors"].str.count(",") + 1
print(n_dirs.value_counts(dropna=False).sort_index())

crew["director_id"] = crew["directors"].str.split(",").str[0]
crew.head()

directors
1       45401
2        3390
3         381
4          91
5          54
6          31
7          18
8          17
9          13
10          8
11         10
12          7
13          4
14          1
15          2
18          1
21          1
26          1
27          1
28          1
31          2
34          1
35          1
36          1
40          1
<NA>       14
Name: count, dtype: Int64


,tconst,directors,director_id
570,tt0000574,nm0846879,nm0846879
2105,tt0002130,"nm0078205,nm0655824,nm0209738",nm0078205
2396,tt0002423,nm0523932,nm0523932
2817,tt0002844,nm0275421,nm0275421
2986,tt0003014,nm0803705,nm0803705


In [14]:
rated = rated.merge(
    crew[["tconst", "director_id"]],
    on="tconst", how="left", validate="one_to_one",
)
print(rated.shape)

(49453, 10)


In [15]:
names = pd.read_csv(
    NAMES,
    **TSV,
    usecols=["nconst", "primaryName"],
    dtype={"nconst": "string[pyarrow]", "primaryName": "string[pyarrow]"},
)
names = names[names["nconst"].isin(rated["director_id"])]
print(names.shape, "unique:", names["nconst"].is_unique)

(20522, 2) unique: True


In [16]:
rated = rated.merge(
    names.rename(columns={"nconst": "director_id", "primaryName": "director"}),
    on="director_id", how="left", validate="many_to_one",
)
del crew, names

print(rated.shape)
print("Missing director name:", rated["director"].isna().sum())
rated[["primaryTitle", "startYear", "director", "averageRating"]].head(10)

(49453, 11)
Missing director name: 15


,primaryTitle,startYear,director,averageRating
0,The Story of the Kelly Gang,1906,Charles Tait,6.0
1,Dante's Inferno,1911,Francesco Bertolini,7.1
2,Passion,1919,Ernst Lubitsch,6.7
3,Fantômas: In the Shadow of the Guillotine,1913,Louis Feuillade,6.9
4,Ingeborg Holm,1913,Victor Sjöström,7.0
5,Fantomas: The Man in Black,1913,Louis Feuillade,6.9
6,Fantômas: The Dead Man Who Killed,1913,Louis Feuillade,6.9
7,The Student of Prague,1913,Hanns Heinz Ewers,6.4
8,The Avenging Conscience: or 'Thou Shalt Not Kill',1914,D.W. Griffith,6.4
9,Cabiria,1914,Giovanni Pastrone,7.1


In [17]:
rated["genre_list"] = rated["genres"].str.split(",")
print(rated[["primaryTitle", "genres", "genre_list"]].head())

rated["genre_list"].explode().value_counts()

                                primaryTitle                      genres  \
0                The Story of the Kelly Gang  Action,Adventure,Biography   
1                            Dante's Inferno     Adventure,Drama,Fantasy   
2                                    Passion     Biography,Drama,Romance   
3  Fantômas: In the Shadow of the Guillotine                 Crime,Drama   
4                              Ingeborg Holm                       Drama   

                       genre_list  
0  [Action, Adventure, Biography]  
1     [Adventure, Drama, Fantasy]  
2     [Biography, Drama, Romance]  
3                  [Crime, Drama]  
4                         [Drama]  


genre_list
Drama          27543
Comedy         16463
Action          8978
Crime           8259
Romance         8224
Thriller        7592
Horror          6693
Adventure       5352
Mystery         4215
Fantasy         2675
Biography       2623
Documentary     2598
Sci-Fi          2394
Family          2074
History         1944
Animation       1644
Music           1498
War             1235
Sport            926
Musical          859
Western          669
Film-Noir        463
News              29
Talk-Show          1
Name: count, dtype: int64

In [18]:
dummies = (rated["genres"]
           .str.get_dummies(sep=",")
           .astype("int8")
           .add_prefix("g_"))
print(dummies.shape)
print(dummies.sum().sort_values(ascending=False))

rated = pd.concat([rated, dummies], axis=1)
print(rated.shape)

(49453, 24)
g_Drama          27543
g_Comedy         16463
g_Action          8978
g_Crime           8259
g_Romance         8224
g_Thriller        7592
g_Horror          6693
g_Adventure       5352
g_Mystery         4215
g_Fantasy         2675
g_Biography       2623
g_Documentary     2598
g_Sci-Fi          2394
g_Family          2074
g_History         1944
g_Animation       1644
g_Music           1498
g_War             1235
g_Sport            926
g_Musical          859
g_Western          669
g_Film-Noir        463
g_News              29
g_Talk-Show          1
dtype: int64
(49453, 36)


In [19]:
(rated.isna().mean() * 100).round(1).sort_values(ascending=False).head(8)

genre_list        0.2
genres            0.2
runtimeMinutes    0.1
tconst            0.0
g_Film-Noir       0.0
g_History         0.0
g_Horror          0.0
g_Music           0.0
dtype: float64

In [20]:
before = len(rated)
rated = rated.dropna(subset=["startYear"])
print("Dropped for missing year:", before - len(rated))
log["5. has a year"] = len(rated)

Dropped for missing year: 0


In [21]:
keep = ["tconst", "primaryTitle", "startYear", "runtimeMinutes",
        "averageRating", "numVotes", "director_id", "director",
        "genres", "genre_list"] + list(dummies.columns)

movies_clean = (
    rated[keep]
    .rename(columns={"primaryTitle": "title", "startYear": "year",
                     "runtimeMinutes": "runtime", "averageRating": "rating",
                     "numVotes": "votes"})
    .sort_values("votes", ascending=False)
    .reset_index(drop=True)
)
assert movies_clean["tconst"].is_unique

print(movies_clean.shape)
movies_clean.head()

(49453, 34)


,tconst,title,year,runtime,rating,votes,director_id,director,genres,genre_list,...,g_Musical,g_Mystery,g_News,g_Romance,g_Sci-Fi,g_Sport,g_Talk-Show,g_Thriller,g_War,g_Western
0,tt0111161,The Shawshank Redemption,1994,142,9.3,3262239,nm0001104,Frank Darabont,Drama,[Drama],...,0,0,0,0,0,0,0,0,0,0
1,tt0468569,The Dark Knight,2008,152,9.1,3248055,nm0634240,Christopher Nolan,"Crime,Thriller","[Crime, Thriller]",...,0,0,0,0,0,0,0,1,0,0
2,tt1375666,Inception,2010,148,8.8,2887178,nm0634240,Christopher Nolan,"Adventure,Sci-Fi,Thriller","[Adventure, Sci-Fi, Thriller]",...,0,0,0,0,1,0,0,1,0,0
3,tt0137523,Fight Club,1999,139,8.8,2681152,nm0000399,David Fincher,"Crime,Drama,Thriller","[Crime, Drama, Thriller]",...,0,0,0,0,0,0,0,1,0,0
4,tt0816692,Interstellar,2014,169,8.7,2630975,nm0634240,Christopher Nolan,"Adventure,Sci-Fi","[Adventure, Sci-Fi]",...,0,0,0,0,1,0,0,0,0,0


In [22]:
os.makedirs(PROCESSED, exist_ok=True)
OUT = f"{PROCESSED}/movies.parquet"
TEST_CSV = f"{PROCESSED}/movies_test.csv"

movies_clean.to_parquet(OUT, index=False)
movies_clean.to_csv(TEST_CSV, index=False)

print("Parquet MB:", round(os.path.getsize(OUT) / 1e6, 2))
print("CSV MB:    ", round(os.path.getsize(TEST_CSV) / 1e6, 2))

Parquet MB: 2.33
CSV MB:     8.02


In [23]:
back = pd.read_parquet(OUT)
print(back[["year", "runtime", "rating", "genre_list"]].dtypes)
print(back.loc[0, "genre_list"])

csv_back = pd.read_csv(TEST_CSV)
print(csv_back[["year", "runtime", "rating", "genre_list"]].dtypes)
print(repr(csv_back.loc[0, "genre_list"]))

os.remove(TEST_CSV)

year            Int16
runtime         Int32
rating        float32
genre_list     object
dtype: object
['Drama']
year            int64
runtime       float64
rating        float64
genre_list        str
dtype: object
"['Drama']"


In [24]:
pd.Series(log).to_frame("rows")

,rows
1. all titles,12947333
2. non-adult movies,751203
3. with any rating,346670
"4. 1,000+ votes",49453
5. has a year,49453


## Cleaning decisions
- Filtered the data from all titles to only non-adult movies
- Selected movies that had 1,000+ votes so as to not skew the results with movies that had few votes